# Num2Text: 숫자·영어·한국어 6방향 Seq2Seq 번역

숫자, 영어 수 표현, 한국어 수 표현을 서로 번역합니다(6방향). 변환 규칙을 직접 작성하지 않고, 제공 데이터의 입력·정답 쌍으로 **GRU Encoder–Decoder + Attention**을 학습하며, 출력은 **Greedy decoding**으로 생성합니다.

- 숫자는 정수로 바꾸지 않고 `"1"`, `"2"`, `"3"` 같은 **문자 토큰**으로 입력합니다.
- 6방향을 **모델 하나**로 처리합니다. 입력 맨 앞의 목표 언어 토큰(`<to_en>`, `<to_ko>`, `<to_num>`)이 번역 방향을 알려 줍니다.
- 영어↔한국어 직접 짝이 부족한 문제를 해결하기 위해, 학습된 모델이 만든 **생성 짝(pseudo-pair)**으로 학습 데이터를 보강했습니다. 자세한 설명과 결과는 아래 `## 생성 짝 보강의 효과`와 `README.md`에 있습니다.

**셀 순서**
1. 구현: 데이터 분할, 토큰 사전, 모델, 학습, 생성 짝, 평가 함수
2. 학습·생성·평가: 세 모델 학습과 test 평가 (실제 실행 출력 포함)
3. 자릿수별 test 결과
4. 저장 모델 로드·테스트 및 사용 방법 (마지막 셀)

원본 데이터를 `data/`에, 세 `.pt` 파일을 이 노트북과 같은 폴더에 두고, 이 폴더를 작업 폴더로 열어 실행하세요.

In [1]:
"""Leakage-safe Num2Text experiment. All numeric model inputs are digit tokens."""
from pathlib import Path
import argparse, collections, copy, csv, hashlib, json, math, random, time
import numpy as np
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

ROOT = Path.cwd()
LANGS = ('num', 'en', 'ko')
DIRECTIONS = [(a,b) for a in LANGS for b in LANGS if a != b]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CONFIG = dict(seed=42, embedding=128, hidden=256, batch_size=512,
              epochs=20, finetune_epochs=8, lr=0.001, pseudo_weight=0.5,
              confidence=0.90, max_decode=64)

def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(8)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def read_data():
    records = {}
    for lang, name in [('en','num2txt.txt'),('ko','num2kor.txt')]:
        for line in (ROOT/'data'/name).read_text(encoding='utf-8-sig').splitlines():
            n, text = line.split('\t', 1)
            n = n.strip()
            if not n or any(c not in "0123456789" for c in n):
                raise ValueError("Invalid digit string")
            text = ' '.join(text.split())
            record = records.setdefault(n, {'num': n})
            assert lang not in record, (n, lang)
            record[lang] = text
    return records

def prepare():
    seed_all(CONFIG['seed'])
    records = read_data()
    # Stratify jointly by digit count and available languages. Split NUMBER IDs once.
    groups = collections.defaultdict(list)
    for n,r in records.items(): groups[(len(n), tuple(sorted(r)))].append(n)
    split = {'train':[], 'val':[], 'test':[]}
    rng = random.Random(CONFIG['seed'])
    for key, ns in sorted(groups.items()):
        ns.sort(key=lambda n:(len(n),n)); rng.shuffle(ns)
        k = max(1, round(len(ns)*0.1)) if len(ns) >= 3 else 0
        split['val'] += ns[:k]; split['test'] += ns[k:2*k]; split['train'] += ns[2*k:]
    sets = {k:set(v) for k,v in split.items()}
    assert not (sets['train'] & sets['val'] or sets['train'] & sets['test'] or sets['val'] & sets['test'])
    assert set.union(*sets.values()) == set(records)
    save_json(ROOT/'results/split_ids.json', split)
    # Vocabulary exclusively learned from original training records.
    tokens = set()
    for n in split['train']:
        for lang,text in records[n].items(): tokens.update(tokenize(text,lang))
    vocab = ['<pad>','<bos>','<eos>','<unk>'] + ['<to_'+l+'>' for l in LANGS] + sorted(tokens)
    save_json(ROOT/'results/vocab.json', vocab)
    audit = {'numbers':len(records), 'split_numbers':{k:len(v) for k,v in split.items()},
             'disjoint_number_ids':True,'vocab_size':len(vocab),
             'source_sha256':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in (ROOT/'data').glob('*.txt')},
             'pairs':{k:dict(collections.Counter(p['direction'] for p in pairs(records,ns))) for k,ns in split.items()}}
    save_json(ROOT/'results/data_audit.json',audit)
    print(json.dumps(audit,ensure_ascii=False),flush=True)
    return records,split,vocab

def tokenize(text,lang):
    return ['en:'+s for s in text.split()] if lang=='en' else [lang+':'+c for c in text]

def pairs(records, ids):
    out=[]
    for n in ids:
        r=records[n]
        for a,b in DIRECTIONS:
            if a in r and b in r:
                out.append(dict(number=n,source=r[a],target=r[b],src=a,tgt=b,direction=a+'2'+b,weight=1.0))
    return out

class Codec:
    def __init__(self,vocab):
        self.vocab=vocab; self.stoi={s:i for i,s in enumerate(vocab)}
    def encode(self,text,lang): return [self.stoi.get(t,3) for t in tokenize(text,lang)]
    def source(self,p): return torch.tensor([self.stoi['<to_'+p['tgt']+'>']]+self.encode(p['source'],p['src'])+[2])
    def target(self,p): return torch.tensor([1]+self.encode(p['target'],p['tgt'])+[2])
    def decode(self,ids,lang):
        out=[]
        for i in ids:
            if i==2: break
            s=self.vocab[i]
            out.append(s.split(':',1)[1] if s.startswith(lang+':') else '<invalid>')
        return ' '.join(out) if lang=='en' else ''.join(out)

def encoded(ps,codec): return [(codec.source(p),codec.target(p),p['weight']) for p in ps]
def batches(data,batch_size,shuffle=False):
    order=np.random.permutation(len(data)) if shuffle else np.arange(len(data))
    for start in range(0,len(data),batch_size):
        rows=[data[i] for i in order[start:start+batch_size]]
        x=pad_sequence([r[0] for r in rows],batch_first=True)
        y=pad_sequence([r[1] for r in rows],batch_first=True)
        yield x.to(DEVICE), y.to(DEVICE),torch.tensor([r[2] for r in rows],device=DEVICE)

class Seq2Seq(nn.Module):
    """GRU encoder + GRU decoder with Luong dot-product attention (no input feeding)."""
    def __init__(self,vocab_size,embedding=128,hidden=256):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,embedding,padding_idx=0)
        self.encoder=nn.GRU(embedding,hidden,batch_first=True)
        self.decoder=nn.GRU(embedding,hidden,batch_first=True)
        self.combine=nn.Linear(hidden*2,hidden)
        self.output=nn.Linear(hidden,vocab_size)
        self.hidden_size=hidden
    def encode(self,x):
        # Packing prevents encoder final state from including right padding.
        lengths=(x!=0).sum(1).cpu()
        packed=pack_padded_sequence(self.embedding(x),lengths,batch_first=True,enforce_sorted=False)
        enc,h=self.encoder(packed)
        enc,_=pad_packed_sequence(enc,batch_first=True,total_length=x.shape[1])
        return enc,h,x==0
    def attend(self,d,enc,mask):
        scores=torch.bmm(d,enc.transpose(1,2))/math.sqrt(self.hidden_size)
        scores=scores.masked_fill(mask[:,None,:],-1e4)
        ctx=torch.bmm(scores.softmax(-1),enc)
        return self.output(torch.tanh(self.combine(torch.cat([d,ctx],-1))))
    def forward(self,x,y):
        enc,h,mask=self.encode(x)
        d,_=self.decoder(self.embedding(y),h)
        return self.attend(d,enc,mask)
    @torch.inference_mode()
    def generate(self,x,max_len=64):
        self.eval(); enc,h,mask=self.encode(x)
        token=torch.ones(x.size(0),1,dtype=torch.long,device=x.device)
        done=torch.zeros(x.size(0),dtype=torch.bool,device=x.device)
        seq=[]; logp=torch.zeros(x.size(0),device=x.device); lens=torch.zeros_like(logp)
        for _ in range(max_len):
            d,h=self.decoder(self.embedding(token),h)
            lp=self.attend(d,enc,mask)[:,0].log_softmax(-1)
            prob,pred=lp.max(-1)
            logp+=prob*(~done); lens+=(~done)
            seq.append(torch.where(done,2,pred))
            done |= pred==2; token=pred[:,None]
            if done.all(): break
        return torch.stack(seq,1).cpu().tolist(),(logp/lens).exp().cpu().tolist(),done.cpu().tolist()

def checkpoint(path,model,vocab,metadata):
    torch.save(dict(state_dict=model.state_dict(),vocab=vocab,config=CONFIG,metadata=metadata),path)

def load_model(path):
    ck=torch.load(path,map_location=DEVICE,weights_only=True)
    model=Seq2Seq(len(ck['vocab']),ck['config']['embedding'],ck['config']['hidden']).to(DEVICE)
    model.load_state_dict(ck['state_dict']); model.eval()
    return model,Codec(ck['vocab']),ck

@torch.inference_mode()
def predictions(model,ps,codec,batch_size=512):
    out=[]
    for start in range(0,len(ps),batch_size):
        chunk=ps[start:start+batch_size]
        x=pad_sequence([codec.source(p) for p in chunk],batch_first=True).to(DEVICE)
        seq,conf,eos=model.generate(x,CONFIG['max_decode'])
        for p,s,c,e in zip(chunk,seq,conf,eos):
            pred=codec.decode(s,p['tgt'])
            out.append({**p,'prediction':pred,'confidence':c,'eos':e,'correct':pred==p.get('target')})
    return out

def metrics(rows):
    groups=collections.defaultdict(list)
    for p in rows:
        groups[(p['direction'],'all')].append(p['correct'])
        groups[(p['direction'],str(len(p['number'])))].append(p['correct'])
    return [dict(direction=d,digits=k,n=len(v),correct=sum(v),exact_match=sum(v)/len(v)) for (d,k),v in sorted(groups.items())]

def train(model,train_pairs,val_pairs,codec,name,epochs,lr,steps_per_epoch=None):
    data=encoded(train_pairs,codec)
    # Fixed, digit-stratified validation subset for inexpensive greedy selection.
    by=collections.defaultdict(list)
    for p in val_pairs: by[(p['direction'],len(p['number']))].append(p)
    rng=random.Random(777); select=[]
    for key,ps in sorted(by.items()): select.extend(rng.sample(ps,min(32,len(ps))))
    optimizer=torch.optim.Adam(model.parameters(),lr=lr)
    best=-1; history=[]
    for epoch in range(1,epochs+1):
        begin=time.time(); model.train(); total=0; count=0
        target_steps=steps_per_epoch or math.ceil(len(data)/CONFIG['batch_size'])
        def epoch_batches():
            step=0
            while step<target_steps:
                for batch in batches(data,CONFIG['batch_size'],True):
                    yield batch
                    step+=1
                    if step>=target_steps: return
        for x,y,w in epoch_batches():
            optimizer.zero_grad(set_to_none=True)
            logits=model(x,y[:,:-1])
            loss=nn.functional.cross_entropy(logits.transpose(1,2),y[:,1:],ignore_index=0,reduction='none')
            per=(loss.sum(1)/(y[:,1:]!=0).sum(1))
            loss=(per*w).sum()/w.sum()
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
            total+=loss.item()*len(x); count+=len(x)
        rows=predictions(model,select,codec)
        mm=metrics(rows); overall=[r['exact_match'] for r in mm if r['digits']=='all']
        score=sum(overall)/len(overall)
        entry=dict(epoch=epoch,loss=total/count,val_macro_exact=score,seconds=time.time()-begin,optimizer_steps=target_steps)
        history.append(entry); print(name,json.dumps(entry),flush=True)
        if score>best:
            best=score; checkpoint(ROOT/f'{name}.pt',model,codec.vocab,dict(name=name,epoch=epoch,val_macro_exact=score))
        save_json(ROOT/'results'/f'{name}_history.json',history)
    best_model,_,_=load_model(ROOT/f'{name}.pt')
    return best_model

def make_pseudo(model,records,split,codec):
    candidates=[]
    for n in split['train']:
        r=records[n]
        if ('en' in r) != ('ko' in r):
            missing='ko' if 'en' in r else 'en'
            candidates.append(dict(number=n,source=n,src='num',tgt=missing,direction='num2'+missing,weight=1.0))
    generated=predictions(model,candidates,codec)
    accepted=[p for p in generated if p['eos'] and p['prediction'] and '<invalid>' not in p['prediction'] and p['confidence']>=CONFIG['confidence']]
    # Additional learned round-trip check. No deterministic number-to-word labels.
    reverse=[dict(number=p['number'],source=p['prediction'],target=p['number'],src=p['tgt'],tgt='num',direction=p['tgt']+'2num',weight=1.) for p in accepted]
    back=predictions(model,reverse,codec)
    kept=[p for p,b in zip(accepted,back) if b['correct'] and b['eos']]
    aug=[]
    for p in kept:
        n=p['number']; known='en' if p['tgt']=='ko' else 'ko'; missing=p['tgt']
        for a,b,src,tgt in [(known,missing,records[n][known],p['prediction']), (missing,known,p['prediction'],records[n][known])]:
            aug.append(dict(number=n,source=src,target=tgt,src=a,tgt=b,direction=a+'2'+b,weight=CONFIG['pseudo_weight']))
    assert set(p['number'] for p in aug)<=set(split['train'])
    save_json(ROOT/'results/pseudo_audit.json',dict(candidates=len(generated),confidence_pass=len(accepted),roundtrip_pass=len(kept),added_pairs=len(aug),threshold=CONFIG['confidence'],no_val_test_numbers=True))
    with (ROOT/'results/pseudo_pairs.jsonl').open('w',encoding='utf-8') as f:
        for p in aug: f.write(json.dumps(p,ensure_ascii=False)+'\n')
    print('PSEUDO',len(aug),flush=True)
    return aug

def run(stage):
    seed_all(CONFIG['seed']); ROOT.joinpath('results').mkdir(exist_ok=True)
    if stage=='baseline':
        records,split,vocab=prepare(); codec=Codec(vocab)
        model=Seq2Seq(len(vocab),CONFIG['embedding'],CONFIG['hidden']).to(DEVICE)
        print('DEVICE',DEVICE,'PARAMETERS',sum(p.numel() for p in model.parameters()),flush=True)
        train(model,pairs(records,split['train']),pairs(records,split['val']),codec,'baseline',CONFIG['epochs'],CONFIG['lr'])
    else:
        records=read_data(); split=json.loads((ROOT/'results/split_ids.json').read_text())
        model,codec,_=load_model(ROOT/'baseline.pt')
        if stage=='augment':
            pseudo=make_pseudo(model,records,split,codec)
            assert pseudo, 'No pseudo pairs passed the predeclared filter.'
            gold=pairs(records,split['train']); val=pairs(records,split['val'])
            seed_all(CONFIG['seed']+1)
            train(model,gold+pseudo,val,codec,'pseudo_augmented',CONFIG['finetune_epochs'],CONFIG['lr']/3)
            # Original-only continuation controls for additional training.
            model,_,_=load_model(ROOT/'baseline.pt'); seed_all(CONFIG['seed']+1)
            train(model,gold,val,codec,'baseline_continued',CONFIG['finetune_epochs'],CONFIG['lr']/3,
                  steps_per_epoch=math.ceil((len(gold)+len(pseudo))/CONFIG['batch_size']))
        elif stage=='evaluate':
            test=pairs(records,split['test']); combined=[]
            for name in ['baseline','baseline_continued','pseudo_augmented']:
                model,codec,_=load_model(ROOT/f'{name}.pt')
                pred=predictions(model,test,codec)
                with (ROOT/'results'/f'{name}_test_predictions.jsonl').open('w',encoding='utf-8') as f:
                    for p in pred: f.write(json.dumps(p,ensure_ascii=False)+'\n')
                for m in metrics(pred): combined.append(dict(model=name,**m))
                print(name,json.dumps([m for m in metrics(pred) if m['digits']=='all']),flush=True)
            with (ROOT/'results/metrics.csv').open('w',newline='',encoding='utf-8-sig') as f:
                w=csv.DictWriter(f,fieldnames=list(combined[0])); w.writeheader(); w.writerows(combined)
            save_json(ROOT/'results/metrics.json',combined)



## 학습·생성·평가

아래 셀은 **전체 재학습**입니다. 세 모델을 차례로 학습해 같은 폴더의 `.pt`를 덮어쓰고, test로 평가합니다. 출력은 이 셀을 실제로 실행한 기록입니다(RTX 4080 기준 약 7분, CPU에서는 훨씬 오래 걸립니다).

- `baseline`: 원본 데이터로 20 epoch 학습
- `pseudo_augmented`: baseline에서 시작해 원본 + 생성 짝으로 8 epoch 추가 학습
- `baseline_continued`: 비교용 대조군. baseline에서 시작해 원본만으로, 생성 짝 모델과 같은 업데이트 수만큼 추가 학습

숫자는 ID 단위로 train/validation/test에 나누고, 토큰 사전은 train 데이터로만 만듭니다. 가중치는 validation으로 고르고, test는 학습이 끝난 뒤에만 평가합니다. 생성 짝은 train 숫자로만 만듭니다.

**저장된 모델만 사용하려면 이 셀을 실행하지 말고, 첫 코드 셀을 실행한 뒤 마지막 셀로 이동하세요.**

In [2]:
run('baseline')
run('augment')
run('evaluate')

{"numbers": 187262, "split_numbers": {"train": 149808, "val": 18727, "test": 18727}, "disjoint_number_ids": true, "vocab_size": 65, "source_sha256": {"num2kor.txt": "d56d80694c56758a9238bc4204cf9fabc8e07a1738c2c804011e6b70f7055f52", "num2txt.txt": "5dfbae2710f4331f15dc4d2d695396c4583d00121c0eff8d8ffff91123c7329c"}, "pairs": {"train": {"num2en": 79998, "num2ko": 79998, "en2num": 79998, "en2ko": 10188, "ko2num": 79998, "ko2en": 10188}, "val": {"num2en": 10001, "num2ko": 10001, "en2num": 10001, "en2ko": 1275, "ko2num": 10001, "ko2en": 1275}, "test": {"num2en": 10001, "num2ko": 10001, "en2num": 10001, "en2ko": 1275, "ko2num": 10001, "ko2en": 1275}}}
DEVICE cuda PARAMETERS 749249
baseline {"epoch": 1, "loss": 0.5236957648349, "val_macro_exact": 0.9218081969629957, "seconds": 8.42877721786499, "optimizer_steps": 665}
baseline {"epoch": 2, "loss": 0.007953046815399288, "val_macro_exact": 0.9644331416777238, "seconds": 7.587127923965454, "optimizer_steps": 665}
baseline {"epoch": 3, "loss": 0.

## 자릿수별 test 결과

원본 데이터로 정답을 확인할 수 있는 test 문제만 평가합니다. 영어↔한국어는 두 파일에 함께 있는 숫자만 정답이 있어서, test에 8자리 이하만 있고(대부분 5자리 이하) **9·10자리 영어↔한국어는 이 표에 포함되지 않습니다.**

In [3]:
for row in json.loads((ROOT/'results/metrics.json').read_text(encoding='utf-8')):
    print(row)

{'model': 'baseline', 'direction': 'en2ko', 'digits': '1', 'n': 1, 'correct': 1, 'exact_match': 1.0}
{'model': 'baseline', 'direction': 'en2ko', 'digits': '2', 'n': 9, 'correct': 9, 'exact_match': 1.0}
{'model': 'baseline', 'direction': 'en2ko', 'digits': '3', 'n': 90, 'correct': 90, 'exact_match': 1.0}
{'model': 'baseline', 'direction': 'en2ko', 'digits': '4', 'n': 900, 'correct': 900, 'exact_match': 1.0}
{'model': 'baseline', 'direction': 'en2ko', 'digits': '5', 'n': 245, 'correct': 245, 'exact_match': 1.0}
{'model': 'baseline', 'direction': 'en2ko', 'digits': '6', 'n': 26, 'correct': 25, 'exact_match': 0.9615384615384616}
{'model': 'baseline', 'direction': 'en2ko', 'digits': '7', 'n': 3, 'correct': 1, 'exact_match': 0.3333333333333333}
{'model': 'baseline', 'direction': 'en2ko', 'digits': '8', 'n': 1, 'correct': 0, 'exact_match': 0.0}
{'model': 'baseline', 'direction': 'en2ko', 'digits': 'all', 'n': 1275, 'correct': 1271, 'exact_match': 0.9968627450980392}
{'model': 'baseline', 'dir

## 생성 짝 보강의 효과

위 원본 test의 6방향 평균은 baseline 99.88%, 대조군 99.93%, 생성 짝 모델 99.93%로 차이가 작습니다. 원본 test의 영어↔한국어 문제가 8자리 이하뿐이라 긴 숫자를 평가하지 못하기 때문입니다.

보강의 목적은 **원본에 직접 짝이 없는 긴 영어↔한국어 번역**입니다. 학습 데이터의 영어↔한국어 직접 짝은 10,188쌍뿐이고 9·10자리는 0쌍이었습니다. test의 9·10자리 숫자 6,000개(양방향 12,000문제)로 별도 시험한 결과는 다음과 같습니다.

| 모델 | 영어→한국어 | 한국어→영어 |
|---|---:|---:|
| baseline | 0 / 6,000 (0.00%) | 0 / 6,000 (0.00%) |
| baseline_continued (원본만 추가 학습) | 0 / 6,000 (0.00%) | 0 / 6,000 (0.00%) |
| pseudo_augmented (생성 짝 보강) | 5,992 / 6,000 (99.87%) | 5,997 / 6,000 (99.95%) |

같은 업데이트 수만큼 원본으로만 더 학습한 대조군은 0%에 머물렀으므로, 향상은 학습량 증가가 아니라 생성 짝 때문입니다.

참고로 숫자를 거쳐 번역하면(영어→숫자→한국어) 보강 없이도 99.8% 이상이 나옵니다. 따라서 보강의 성과는 **숫자를 거치지 않는 직접 번역 모델 하나로 경유 방식과 같은 수준에 도달한 것**입니다.

이 별도 시험은 학습이 끝난 뒤 `extra_eval.py`로 실행합니다. 빠진 쪽 정답은 원본 20만 줄과 모두 일치하는 수 읽기 규칙으로 만들고, 규칙은 채점용 정답지에만 씁니다(학습·생성 짝·가중치 선택에는 사용하지 않음). 자릿수별 결과와 숫자 경유 비교, validation 구성 이유는 `README.md`에 있습니다.

## 저장 모델 로드·테스트 및 사용 방법
아래 마지막 셀에 사용법과 실제 로드·추론 코드를 포함합니다.

In [4]:
# 사용법: 숫자도 문자열로 입력합니다. 언어 코드는 num, en, ko입니다.
# 예: translate("12345", "num", "en")
# 모델 파일은 이 노트북과 같은 폴더에 둡니다.
model, codec, saved = load_model(ROOT/'pseudo_augmented.pt')
def translate(text, source_language, target_language):
    if not isinstance(text,str):
        raise ValueError('입력은 문자열이어야 합니다.')
    if source_language not in LANGS or target_language not in LANGS or source_language==target_language:
        raise ValueError('지원하지 않는 번역 방향입니다.')
    text=' '.join(text.split())
    if not text:
        raise ValueError('빈 입력은 허용하지 않습니다.')
    if source_language=='num':
        if any(c not in '0123456789' for c in text) or len(text)>10 or (len(text)>1 and text[0]=='0'):
            raise ValueError('0~9로 구성한 최대 10자리 숫자를 입력하세요. 선행 0은 허용하지 않습니다.')
    if 3 in codec.encode(text,source_language):
        raise ValueError('학습 사전에 없는 입력 토큰입니다.')
    p=dict(number='',source=text,src=source_language,tgt=target_language,direction=source_language+'2'+target_language,weight=1.)
    result=predictions(model,[p],codec)[0]
    if not result['eos']:
        raise ValueError('최대 길이 안에 출력이 종료되지 않았습니다.')
    return result['prediction']
print('Loaded:',saved['metadata'])
records=read_data()
split=json.loads((ROOT/'results/split_ids.json').read_text(encoding='utf-8'))
example=next(n for n in split['test'] if len(records[n])==3 and len(n)>=4)
for p in pairs(records,[example]):
    pred=translate(p['source'],p['src'],p['tgt'])
    print(p['direction'],p['source'],'→',pred,'gold=',p['target'],'correct=',pred==p['target'])
for invalid in ['012','12x','１２','12345678901','']:
    try: translate(invalid,'num','en')
    except ValueError as error: print('입력 검사:',repr(invalid),str(error))
    else: raise RuntimeError('입력 검사 실패')


Loaded: {'name': 'pseudo_augmented', 'epoch': 3, 'val_macro_exact': 1.0}
num2en 8990 → eight thousand nine hundred ninety gold= eight thousand nine hundred ninety correct= True
num2ko 8990 → 팔천구백구십 gold= 팔천구백구십 correct= True
en2num eight thousand nine hundred ninety → 8990 gold= 8990 correct= True
en2ko eight thousand nine hundred ninety → 팔천구백구십 gold= 팔천구백구십 correct= True
ko2num 팔천구백구십 → 8990 gold= 8990 correct= True
ko2en 팔천구백구십 → eight thousand nine hundred ninety gold= eight thousand nine hundred ninety correct= True
입력 검사: '012' 0~9로 구성한 최대 10자리 숫자를 입력하세요. 선행 0은 허용하지 않습니다.
입력 검사: '12x' 0~9로 구성한 최대 10자리 숫자를 입력하세요. 선행 0은 허용하지 않습니다.
입력 검사: '１２' 0~9로 구성한 최대 10자리 숫자를 입력하세요. 선행 0은 허용하지 않습니다.
입력 검사: '12345678901' 0~9로 구성한 최대 10자리 숫자를 입력하세요. 선행 0은 허용하지 않습니다.
입력 검사: '' 빈 입력은 허용하지 않습니다.
